## Embeddings

Dalam contoh kami sebelumnya, kami bekerja dengan vektor bag-of-words berdimensi tinggi dengan panjang `vocab_size`, dan kami secara eksplisit mengonversi dari vektor representasi posisi berdimensi rendah ke representasi one-hot yang jarang. Representasi one-hot ini tidak efisien dalam penggunaan memori, selain itu, setiap kata diperlakukan secara terpisah satu sama lain, artinya vektor yang di-encode one-hot tidak menyatakan kesamaan semantik antara kata.

Dalam unit ini, kami akan melanjutkan eksplorasi dataset **News AG**. Untuk memulai, mari muat data dan dapatkan beberapa definisi dari notebook sebelumnya.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Apa itu embedding?

Ide dari **embedding** adalah merepresentasikan kata-kata dengan vektor padat berdimensi rendah, yang dengan cara tertentu mencerminkan makna semantik suatu kata. Nanti kita akan membahas bagaimana membangun embedding kata yang bermakna, tapi untuk sekarang mari kita anggap embedding sebagai cara untuk menurunkan dimensi sebuah vektor kata.

Jadi, lapisan embedding akan mengambil sebuah kata sebagai input, dan menghasilkan vektor keluaran dengan dimensi `embedding_size` yang ditentukan. Dalam arti tertentu, lapisan ini sangat mirip dengan lapisan `Linear`, tetapi alih-alih mengambil vektor one-hot encoded, lapisan ini dapat mengambil nomor kata sebagai input.

Dengan menggunakan lapisan embedding sebagai lapisan pertama dalam jaringan kita, kita dapat beralih dari model bag-of-words ke model **embedding bag**, di mana kita pertama mengonversi setiap kata dalam teks kita menjadi embedding yang sesuai, dan kemudian menghitung fungsi agregasi pada semua embedding tersebut, seperti `sum`, `average` atau `max`.

![Gambar yang menunjukkan pengklasifikasi embedding untuk lima kata dalam urutan.](../../../../../translated_images/id/embedding-classifier-example.b77f021a7ee67eee.webp)

Jaringan neural pengklasifikasi kita akan dimulai dengan lapisan embedding, kemudian lapisan agregasi, dan pengklasifikasi linier di atasnya:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Menangani ukuran urutan yang bervariasi

Sebagai hasil dari arsitektur ini, minibatch untuk jaringan kami perlu dibuat dengan cara tertentu. Pada unit sebelumnya, saat menggunakan bag-of-words, semua tensor BoW dalam sebuah minibatch memiliki ukuran yang sama `vocab_size`, terlepas dari panjang sebenarnya dari urutan teks kami. Setelah kita beralih ke word embeddings, jumlah kata dalam setiap sampel teks akan bervariasi, dan saat menggabungkan sampel-sampel tersebut ke dalam minibatch, kita harus menerapkan padding.

Ini dapat dilakukan dengan menggunakan teknik yang sama yaitu menyediakan fungsi `collate_fn` ke datasource:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Melatih pengklasifikasi embedding

Sekarang setelah kita telah mendefinisikan dataloader yang tepat, kita dapat melatih model menggunakan fungsi pelatihan yang telah kita definisikan di unit sebelumnya:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Catatan**: Kita hanya melatih untuk 25 ribu data di sini (kurang dari satu epoch penuh) demi waktu, tetapi Anda dapat melanjutkan pelatihan, menulis fungsi untuk melatih selama beberapa epoch, dan bereksperimen dengan parameter learning rate untuk mencapai akurasi yang lebih tinggi. Anda harus dapat mencapai akurasi sekitar 90%.


### Lapisan EmbeddingBag dan Representasi Urutan dengan Panjang Variabel

Dalam arsitektur sebelumnya, kita perlu mengisi semua urutan agar memiliki panjang yang sama agar bisa dimasukkan ke dalam minibatch. Ini bukan cara paling efisien untuk merepresentasikan urutan dengan panjang variabel - pendekatan lain adalah menggunakan vektor **offset**, yang memuat offset dari semua urutan yang disimpan dalam satu vektor besar.

![Gambar yang menunjukkan representasi urutan offset](../../../../../translated_images/id/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Catatan**: Pada gambar di atas, kami menunjukkan urutan karakter, tetapi dalam contoh kami bekerja dengan urutan kata. Namun, prinsip umum merepresentasikan urutan dengan vektor offset tetap sama.

Untuk bekerja dengan representasi offset, kami menggunakan lapisan [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Ini mirip dengan `Embedding`, tetapi mengambil vektor konten dan vektor offset sebagai input, dan juga menyertakan lapisan pengambilan rata-rata, yang dapat berupa `mean`, `sum` atau `max`.

Berikut adalah jaringan yang dimodifikasi yang menggunakan `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Untuk mempersiapkan dataset untuk pelatihan, kita perlu menyediakan fungsi konversi yang akan mempersiapkan vektor offset:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Perhatikan, bahwa berbeda dengan semua contoh sebelumnya, jaringan kami sekarang menerima dua parameter: vektor data dan vektor offset, yang memiliki ukuran berbeda. Demikian pula, pemuat data kami juga menyediakan 3 nilai bukannya 2: baik vektor teks maupun offset disediakan sebagai fitur. Oleh karena itu, kami perlu sedikit menyesuaikan fungsi pelatihan kami untuk mengatasi hal tersebut:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Embedding Semantik: Word2Vec

Dalam contoh sebelumnya, lapisan embedding model belajar memetakan kata ke representasi vektor, namun, representasi ini tidak memiliki makna semantik yang signifikan. Akan sangat baik untuk belajar representasi vektor sedemikian rupa, sehingga kata-kata yang mirip atau sinonim akan sesuai dengan vektor yang dekat satu sama lain dalam hal jarak vektor tertentu (misalnya jarak euclidean).

Untuk melakukan itu, kita perlu melatih model embedding kita terlebih dahulu pada kumpulan teks besar dengan cara tertentu. Salah satu cara pertama untuk melatih embedding semantik disebut [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Ini didasarkan pada dua arsitektur utama yang digunakan untuk menghasilkan representasi terdistribusi dari kata-kata:

 - **Continuous bag-of-words** (CBoW) — dalam arsitektur ini, kita melatih model untuk memprediksi sebuah kata dari konteks sekitarnya. Diberikan ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, tujuan model adalah memprediksi $W_0$ dari $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** adalah kebalikan dari CBoW. Model menggunakan jendela kata-kata konteks di sekitarnya untuk memprediksi kata saat ini.

CBoW lebih cepat, sementara skip-gram lebih lambat, tapi melakukan pekerjaan yang lebih baik dalam merepresentasikan kata-kata yang jarang muncul.

![Gambar yang menunjukkan kedua algoritma CBoW dan Skip-Gram untuk mengubah kata menjadi vektor.](../../../../../translated_images/id/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Untuk bereksperimen dengan embedding word2vec yang telah dilatih sebelumnya pada dataset Google News, kita dapat menggunakan pustaka **gensim**. Di bawah ini kita temukan kata-kata yang paling mirip dengan 'neural'

> **Catatan:** Saat pertama kali Anda membuat vektor kata, mengunduhnya dapat membutuhkan waktu!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Kita juga dapat menghitung embedding vektor dari kata tersebut, untuk digunakan dalam melatih model klasifikasi (kami hanya menampilkan 20 komponen pertama dari vektor untuk kejelasan):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Hal hebat tentang embedding semantik adalah Anda dapat memanipulasi pengkodean vektor untuk mengubah semantik. Misalnya, kita dapat meminta untuk menemukan sebuah kata, yang representasi vektornya sedekat mungkin dengan kata *king* dan *woman*, dan sejauh mungkin dari kata *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Baik CBoW maupun Skip-Grams adalah embedding “prediktif”, karena mereka hanya memperhitungkan konteks lokal. Word2Vec tidak memanfaatkan konteks global. 

**FastText**, membangun dari Word2Vec dengan mempelajari representasi vektor untuk setiap kata dan karakter n-gram yang ditemukan dalam setiap kata. Nilai representasi tersebut kemudian dirata-ratakan menjadi satu vektor pada setiap langkah pelatihan. Meskipun ini menambah banyak komputasi tambahan pada pra-pelatihan, metode ini memungkinkan embedding kata untuk mengkode informasi sub-kata. 

Metode lain, **GloVe**, memanfaatkan ide matriks ko-ocurrence, menggunakan metode neural untuk mengurai matriks ko-ocurrence menjadi vektor kata yang lebih ekspresif dan non-linear.

Anda dapat mencoba contoh dengan mengubah embedding menjadi FastText dan GloVe, karena gensim mendukung beberapa model embedding kata yang berbeda.


## Menggunakan Embedding Pra-Latih di PyTorch

Kita dapat memodifikasi contoh di atas untuk mengisi matriks di lapisan embedding kita dengan embedding semantik, seperti Word2Vec. Kita perlu memperhatikan bahwa kosakata embedding pra-latih dan korpus teks kita kemungkinan tidak akan cocok, maka kita akan menginisialisasi bobot untuk kata-kata yang hilang dengan nilai acak:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Sekarang mari kita latih model kita. Perlu dicatat bahwa waktu yang dibutuhkan untuk melatih model jauh lebih lama dibandingkan contoh sebelumnya, karena ukuran lapisan embedding yang lebih besar, dan dengan demikian jumlah parameter yang jauh lebih banyak. Juga, karena hal ini, kita mungkin perlu melatih model kita dengan lebih banyak contoh jika kita ingin menghindari overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Dalam kasus kami, kami tidak melihat peningkatan akurasi yang besar, yang kemungkinan disebabkan oleh perbedaan kosakata yang cukup berbeda.
Untuk mengatasi masalah kosakata yang berbeda, kami dapat menggunakan salah satu solusi berikut:
* Melatih ulang model word2vec pada kosakata kami
* Memuat dataset kami dengan kosakata dari model word2vec yang sudah dilatih. Kosakata yang digunakan untuk memuat dataset dapat ditentukan saat pemuatan.

Pendekatan yang terakhir tampaknya lebih mudah, terutama karena kerangka kerja PyTorch `torchtext` sudah memiliki dukungan bawaan untuk embeddings. Kita dapat, misalnya, menginisialisasi kosakata berbasis GloVe dengan cara berikut:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Kosa kata yang dimuat memiliki operasi dasar sebagai berikut:
* `vocab.stoi` kamus memungkinkan kita mengubah kata menjadi indeks kamusnya
* `vocab.itos` melakukan kebalikan - mengubah nomor menjadi kata
* `vocab.vectors` adalah array vektor embedding, jadi untuk mendapatkan embedding sebuah kata `s` kita perlu menggunakan `vocab.vectors[vocab.stoi[s]]`

Berikut adalah contoh manipulasi embedding untuk menunjukkan persamaan **kind-man+woman = queen** (Saya harus mengatur sedikit koefisien agar berhasil):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Untuk melatih pengklasifikasi menggunakan embedding tersebut, pertama-tama kita perlu mengenkode dataset kita menggunakan kosakata GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Seperti yang telah kita lihat di atas, semua embedding vektor disimpan dalam matriks `vocab.vectors`. Hal ini membuatnya sangat mudah untuk memuat bobot-bobot tersebut ke dalam bobot lapisan embedding menggunakan penyalinan sederhana:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Sekarang mari kita latih model kita dan lihat apakah kita mendapatkan hasil yang lebih baik:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Salah satu alasan mengapa kita tidak melihat peningkatan akurasi yang signifikan adalah karena beberapa kata dari dataset kita tidak ada dalam kosa kata GloVe yang sudah dilatih sebelumnya, dan dengan demikian kata-kata tersebut pada dasarnya diabaikan. Untuk mengatasi hal ini, kita dapat melatih embedding kita sendiri pada dataset kita. 


## Embedding Kontekstual

Salah satu keterbatasan utama dari representasi embedding yang dipra-latih tradisional seperti Word2Vec adalah masalah disambiguasi makna kata. Walaupun embedding yang dipra-latih dapat menangkap sebagian makna kata dalam konteks, setiap kemungkinan makna dari sebuah kata dikodekan dalam embedding yang sama. Ini dapat menyebabkan masalah dalam model hilir, karena banyak kata seperti kata 'play' memiliki makna yang berbeda tergantung konteks penggunaannya.

Misalnya kata 'play' dalam dua kalimat berbeda ini memiliki makna yang cukup berbeda:
- Saya pergi menonton **play** di teater.
- John ingin **play** dengan teman-temannya.

Embedding yang dipra-latih di atas merepresentasikan kedua makna kata 'play' tersebut dalam embedding yang sama. Untuk mengatasi keterbatasan ini, kita perlu membuat embedding berdasarkan **model bahasa**, yang dilatih pada korpus teks yang besar, dan *mengetahui* bagaimana kata-kata dapat disusun dalam konteks yang berbeda. Pembahasan tentang embedding kontekstual di luar cakupan tutorial ini, tetapi kita akan kembali membahasnya saat membicarakan model bahasa di unit berikutnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
